In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score


from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from imblearn.over_sampling import SMOTE

In [ ]:

!pip install opendatasets --upgrade --quiet
import opendatasets as od

od.download("https://www.kaggle.com/datasets/ealaxi/paysim1")

Skipping, found downloaded files in "./paysim1" (use force=True to force download)


In [ ]:

df = pd.read_csv("paysim1/PS_20174392719_1491204439457_log.csv")

print("Dataset Shape (Rows, Columns):", df.shape)

Dataset Shape (Rows, Columns): (6362620, 11)


In [ ]:
print("--- MISSING VALUES ---")
print(df.isnull().sum())

print("\n--- CLASS DISTRIBUTION (isFraud) ---")
print(df['isFraud'].value_counts())

--- MISSING VALUES ---
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

--- CLASS DISTRIBUTION (isFraud) ---
isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [ ]:
#  filtered table with only TRANSFER and CASH_OUT
df_clean = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

# remaining row count
print("Remaining rows:", len(df_clean))

Remaining rows: 2770409


In [ ]:
df_clean.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'], inplace=True, errors='ignore')
# Calculate the exact mathematical error in the sender's account
df_clean['errorBalanceOrig'] = df_clean['newbalanceOrig'] + df_clean['amount'] - df_clean['oldbalanceOrg']

# Calculate the exact mathematical error in the receiver's account
df_clean['errorBalanceDest'] = df_clean['oldbalanceDest'] + df_clean['amount'] - df_clean['newbalanceDest']
df_clean.shape

(2770409, 10)

In [ ]:
# CELL 7: Translate text categories into numbers
encoder = LabelEncoder()
df_clean['type'] = encoder.fit_transform(df_clean['type'])

# Print out the data types of every column to prove zero text remains
print("--- COLUMN DATA TYPES ---")
print(df_clean.dtypes)
df_clean.head()

--- COLUMN DATA TYPES ---
step                  int64
type                  int64
amount              float64
oldbalanceOrg       float64
newbalanceOrig      float64
oldbalanceDest      float64
newbalanceDest      float64
isFraud               int64
errorBalanceOrig    float64
errorBalanceDest    float64
dtype: object


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,errorBalanceOrig,errorBalanceDest
2,1,1,181.00,181.0,0.0,0.0,0.00,1,0.00,181.0
3,1,0,181.00,181.0,0.0,21182.0,0.00,1,0.00,21363.0
15,1,0,229133.94,15325.0,0.0,5083.0,51513.44,0,213808.94,182703.5
19,1,1,215310.30,705.0,0.0,22425.0,0.00,0,214605.30,237735.3
24,1,1,311685.89,10835.0,0.0,6267.0,2719172.89,0,300850.89,-2401220.0


In [ ]:
# Define features (X) and target (y)
X = df_clean.drop('isFraud', axis=1)
y = df_clean['isFraud']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# CELL 9: Balance the Training Set using SMOTE

print("--- BEFORE SMOTE (Training Set) ---")
print(y_train.value_counts())

# 1. Initialize the SMOTE tool
smote = SMOTE( sampling_strategy=0.1, random_state=42)

# 2. Generate synthetic fraud examples ONLY in the training set
print("\nRunning SMOTE... (This might take 30-60 seconds on 2.2 million rows)...")
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("\n--- AFTER SMOTE (Balanced Training Set) ---")
print(y_train_balanced.value_counts())

--- BEFORE SMOTE (Training Set) ---
isFraud
0    1933537
1       5749
Name: count, dtype: int64

Running SMOTE... (This might take 30-60 seconds on 2.2 million rows)...

--- AFTER SMOTE (Balanced Training Set) ---
isFraud
0    1933537
1     193353
Name: count, dtype: int64


In [ ]:
from sklearn.preprocessing import StandardScaler

# CELL 8: Scale the Data

scaler = StandardScaler()

X_train_balanced = scaler.fit_transform(X_train_balanced)

X_test = scaler.transform(X_test)

In [ ]:
# CELL 10: Train KNN on the Balanced Data
from sklearn.neighbors import KNeighborsClassifier
import time
import joblib # Import joblib for saving and loading models

print("Initializing training")

model = KNeighborsClassifier(
        n_neighbors= 5
)

print("Training started")
start_time = time.time()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_train_balanced:", X_train_balanced.shape)
print("y_train_balanced:", y_train_balanced.shape)

# 2. Train the model
model.fit(X_train_balanced, y_train_balanced)

end_time = time.time()
print(f"\n--- TRAINING COMPLETE! ---")
print(f"Time taken: {round(end_time - start_time, 2)} seconds")


# Export Production Model
print("\nExporting production model...")

joblib.dump(model, "KNN_model.joblib")
joblib.dump(encoder, "label_encoder.joblib")
joblib.dump(X.columns.tolist(), "feature_columns.joblib")

print("✅ KNN_model.joblib")
print("✅ label_encoder.joblib")
print("✅ feature_columns.joblib")
print("\nProduction model exported successfully!")


Initializing training
Training started
X_train: (1939286, 9)
y_train: (1939286,)
X_train_balanced: (2126890, 9)
y_train_balanced: (2126890,)

--- TRAINING COMPLETE! ---
Time taken: 21.53 seconds
Your Random Forest model is officially trained and ready to hunt fraudsters!

Exporting production model...
✅ KNN_model.joblib
✅ label_encoder.joblib
✅ feature_columns.joblib

Production model exported successfully!


In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

def evaluate_model(model, X_eval, y_eval):
    # Predictions
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1]

    print("=" * 60)
    print("MODEL EVALUATION")
    print("=" * 60)

    print(f"Evaluating rows: {len(y_eval)}\n")

    print(f"Precision : {precision_score(y_eval, y_pred):.6f}")
    print(f"Recall    : {recall_score(y_eval, y_pred):.6f}")
    print(f"F1 Score  : {f1_score(y_eval, y_pred):.6f}")
    print(f"ROC-AUC   : {roc_auc_score(y_eval, y_prob):.6f}")
    print(f"PR-AUC    : {average_precision_score(y_eval, y_prob):.6f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_eval, y_pred))

    print("\nClassification Report:")
    print(classification_report(
        y_eval,
        y_pred,
        target_names=["Legitimate", "Fraud"]
    ))

    # FP / FN count
    cm = confusion_matrix(y_eval, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print("\nFalse Positives :", fp)
    print("False Negatives :", fn)

    return {
        "precision": precision_score(y_eval, y_pred),
        "recall": recall_score(y_eval, y_pred),
        "f1": f1_score(y_eval, y_pred),
        "roc_auc": roc_auc_score(y_eval, y_prob),
        "pr_auc": average_precision_score(y_eval, y_prob),
        "false_positives": fp,
        "false_negatives": fn
    }

results = evaluate_model(
    model,
    X_test,
    y_test
)

print("\n--- RESULTS ---")
print(results)



MODEL EVALUATION
Evaluating rows: 831123

Precision : 0.390886
Recall    : 0.922484
F1 Score  : 0.549100
ROC-AUC   : 0.979428
PR-AUC    : 0.688722

Confusion Matrix:
[[825117   3542]
 [   191   2273]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00    828659
       Fraud       0.39      0.92      0.55      2464

    accuracy                           1.00    831123
   macro avg       0.70      0.96      0.77    831123
weighted avg       1.00      1.00      1.00    831123


False Positives : 3542
False Negatives : 191

--- RESULTS ---
{'precision': 0.39088564058469477, 'recall': 0.9224837662337663, 'f1': 0.5491001328662882, 'roc_auc': np.float64(0.9794278952617909), 'pr_auc': np.float64(0.6887224933120901), 'false_positives': np.int64(3542), 'false_negatives': np.int64(191)}
